In [ ]:
import pandas as pd 
import numpy as np 
from datetime import datetime, timedelta
import os 
import json


In [ ]:

np.random.seed(42)

CONFIG = {
    "n_users": 500,
    "n_devices": 1000,
    "simulation_days": 30,
    "start_date": "2026-06-01",
    "output_dir": "../data"
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)

def generate_users(n_users):
    print("Generating Users...")

    roles = ["Standard_User", "Admin", "Privileged_User"]
    role_probs = [0.80, 0.05, 0.15]

    departments = ["IT", "Finance", "HR", "Engineering", "Executive"]
    dept_probs = [0.30, 0.20, 0.10, 0.35, 0.05]

    users = pd.DataFrame({
        "user_id": [f"USR-{str(i).zfill(4)}" for i in range(1, n_users + 1)],
        "username": [f"user{i}" for i in range(1, n_users + 1)],
        "role": np.random.choice(roles, size=n_users, p=role_probs),
        "department": np.random.choice(departments, size=n_users, p=dept_probs),
        "vpn_flag": np.random.choice([True, False], size=n_users, p=[0.4, 0.6])
    })

    users["typical_login_hours"] = np.where(
        users["department"].isin(["Engineering", "IT"]),
        np.random.choice([7, 8, 9, 22], size=n_users, p=[0.4, 0.3, 0.2, 0.1]),
        np.random.choice([7, 8, 9], size=n_users, p=[0.3, 0.4, 0.3])    
    )

    users["typical_login_end"] = (users["typical_login_hours"] + 9) % 24

    start_dt = datetime(2021, 1, 1)
    end_dt = datetime(2026, 5, 31)
    delta_days = (end_dt - start_dt).days
    users["hire_date"] = [start_dt + timedelta(days=int(np.random.randint(0, delta_days))) for _ in range(n_users)]
    print(f"Generated {len(users)} users")
    return users 
    
def generate_devices(n_devices, users_df):
    print("Generating devices table")

    device_types = ["Laptop", "Desktop", "Server", "Database_Server", "IoT_Sensor"]
    dt_probs = [0.50, 0.20, 0.15, 0.10, 0.05]
    os_versions = ["Windows_11", "Ubuntu_22.04", "macOS_14", "CentOS_7"]

    devices = pd.DataFrame({
        "device_id": [f"DEV-{str(i).zfill(4)}" for i in range(1, n_devices + 1)],
        # Randomly assign a user_id from the users table
        "user_id": np.random.choice(users_df["user_id"], size=n_devices),
        "device_type": np.random.choice(device_types, size=n_devices, p=dt_probs),
        "os_version": np.random.choice(os_versions, size=n_devices)
    })

    is_server = devices["device_type"].isin(["Server", "Database_Server"])

    devices["criticality_score"] = np.where(
        is_server,
        np.random.randint(6, 11, size=n_devices),
        np.random.randint(1, 6, size=n_devices)
    ) 

    def generate_ip():
        subnet = np.random.choice(["10.0", "192.168"])
        return f"{subnet}.{np.random.randint(0, 10)}.{np.random.randint(1, 255)}"

    devices["ip_address"] = [generate_ip() for _ in range(n_devices)]

    devices = devices.merge(users_df[["user_id", "department"]], on="user_id", how="left")

    print(f" Generated {len(devices)} devices.")
    return devices

users_df = generate_users(CONFIG["n_users"])
devices_df = generate_devices(CONFIG["n_devices"], users_df)

display(users_df.head())
display(devices_df.head())


Generating Users...
Generated 500 users
Generating devices table
 Generated 1000 devices.


,user_id,username,role,department,vpn_flag,typical_login_hours,typical_login_end,hire_date
0,USR-0001,user1,Standard_User,Engineering,True,8,17,2026-05-25
1,USR-0002,user2,Privileged_User,HR,False,7,16,2025-09-07
2,USR-0003,user3,Standard_User,Finance,False,9,18,2025-11-10
3,USR-0004,user4,Standard_User,Engineering,False,7,16,2023-02-11
4,USR-0005,user5,Standard_User,Engineering,False,7,16,2023-06-22


,device_id,user_id,device_type,os_version,criticality_score,ip_address,department
0,DEV-0001,USR-0241,Desktop,Ubuntu_22.04,1,10.0.6.106,Finance
1,DEV-0002,USR-0370,Server,Windows_11,9,192.168.9.232,Engineering
2,DEV-0003,USR-0005,Server,macOS_14,8,10.0.8.98,Engineering
3,DEV-0004,USR-0084,Server,CentOS_7,6,10.0.6.119,Finance
4,DEV-0005,USR-0308,Laptop,Windows_11,1,192.168.8.152,Engineering


In [7]:
# ==========================================
# STEP 3: HIDDEN COMPROMISE STATE TIMELINE
# ==========================================
print("Initializing hidden compromise state timeline...")

total_hours = CONFIG["simulation_days"] * 24
n_devices = CONFIG["n_devices"]

# Create a matrix: rows = devices, columns = hours. All start at 0 (Safe).
# C_matrix shape: (1000, 720)
C_matrix = np.zeros((n_devices, total_hours), dtype=int)

# We will store the episode_type in a separate matrix
# 0 = Safe, 1 = Malicious (TP), 2 = Authorized (BTP)
episode_matrix = np.zeros((n_devices, total_hours), dtype=int)

# Seed ~15 attack episodes on random devices at random times
n_attacks = 15
attack_devices = np.random.choice(n_devices, size=n_attacks, replace=False)
attack_start_hours = np.random.randint(0, total_hours - 100, size=n_attacks)

for i in range(n_attacks):
    dev_idx = attack_devices[i]
    start_h = attack_start_hours[i]
    
    # Dwell time: LogNormal(mu=4.5, sigma=1.0) -> median ~90 hours
    dwell_time = int(np.random.lognormal(mean=4.5, sigma=1.0))
    end_h = min(start_h + dwell_time, total_hours) # Cap at 720 hours
    
    # Flip the compromise state to 1
    C_matrix[dev_idx, start_h:end_h] = 1
    
    # Assign episode type: 65% Malicious (1), 35% Authorized (2)
    ep_type = np.random.choice([1, 2], p=[0.65, 0.35])
    episode_matrix[dev_idx, start_h:end_h] = ep_type

print(f"✅ Seeded {n_attacks} compromise episodes.")

# ==========================================
# STEP 4: GENERATE LOG_EVENTS TABLE
# ==========================================
print("Generating log events (this may take a few seconds)...")

# 4.1 Calculate Poisson Lambda for the entire matrix
# Safe: 10 events/hr (C=0), Compromised: 100 events/hr (C=1)
lambda_matrix = 10 * (1 + 9 * C_matrix)

# 4.2 Generate event counts for every device-hour at once
# events_matrix shape: (1000, 720)
events_matrix = np.random.poisson(lambda_matrix)
total_events = events_matrix.sum()
print(f"Generating {total_events:,} individual log rows...")

# 4.3 Flatten the matrix into a 1D list of events
# np.repeat creates an array of device indices and hours repeated by the event count
device_indices = np.repeat(np.arange(n_devices), events_matrix.sum(axis=1))
hour_indices = np.repeat(np.tile(np.arange(total_hours), n_devices), events_matrix.flatten())

# Filter out devices/hours with 0 events (this shrinks the arrays to exactly total_events)
# A quick trick to align them:
device_indices = np.repeat(np.arange(n_devices), events_matrix.sum(axis=1))
hour_indices = np.concatenate([np.repeat(np.arange(total_hours), events_matrix[i]) for i in range(n_devices)])

# 4.4 Create the base DataFrame
log_events = pd.DataFrame({
    "device_idx": device_indices,
    "hour_offset": hour_indices
})

# Map device_idx back to device_id
device_ids = devices_df["device_id"].values
log_events["device_id"] = device_ids[log_events["device_idx"].values]

# Get the hidden state (C_t) and episode type for each event
log_events["hidden_compromise_state"] = C_matrix[log_events["device_idx"].values, log_events["hour_offset"].values]
log_events["hidden_episode_type"] = episode_matrix[log_events["device_idx"].values, log_events["hour_offset"].values]

# 4.5 Generate Timestamps with ±3 second jitter
start_date = datetime.strptime(CONFIG["start_date"], "%Y-%m-%d")
# Base time: hour offset * 3600 seconds
base_seconds = log_events["hour_offset"] * 3600
# Jitter: uniform random between 0 and 3600 seconds (within the hour)
jitter = np.random.uniform(0, 3600, size=len(log_events))
log_events["timestamp"] = start_date + pd.to_timedelta(base_seconds + jitter, unit="s")

# 4.6 Generate Bytes Transferred (Log-Normal Mixture)
# Vectorized mixture logic
is_safe = log_events["hidden_compromise_state"] == 0
is_comp = ~is_safe

bytes_arr = np.zeros(len(log_events))

# Safe state: 90% LogNormal(11, 1.0), 10% LogNormal(16, 1.0)
safe_mask_1 = is_safe & (np.random.rand(len(log_events)) < 0.90)
safe_mask_2 = is_safe & (~safe_mask_1)
bytes_arr[safe_mask_1] = np.random.lognormal(mean=11, sigma=1.0, size=safe_mask_1.sum())
bytes_arr[safe_mask_2] = np.random.lognormal(mean=16, sigma=1.0, size=safe_mask_2.sum())

# Compromised state: 55% LogNormal(19, 1.5) [Bulk Exfil], 45% LogNormal(13, 1.5) [Low/Slow]
comp_mask_1 = is_comp & (np.random.rand(len(log_events)) < 0.55)
comp_mask_2 = is_comp & (~comp_mask_1)
bytes_arr[comp_mask_1] = np.random.lognormal(mean=19, sigma=1.5, size=comp_mask_1.sum())
bytes_arr[comp_mask_2] = np.random.lognormal(mean=13, sigma=1.5, size=comp_mask_2.sum())

log_events["bytes_transferred"] = bytes_arr

# 4.7 Generate Categorical Features (Process Names & Geo Locations) - CORRECTED
normal_processes = ["chrome.exe", "outlook.exe", "teams.exe", "excel.exe", "svchost.exe"]
suspicious_processes = ["powershell.exe", "cmd.exe", "wscript.exe"]
malicious_processes = ["mimikatz.exe", "psexec.exe", "sharphound.exe"]

normal_geos = ["PH", "SG", "US", "DE"]
anomalous_geos = ["RU", "CN", "Unknown_Proxy"]

# Default to normal
log_events["process_name"] = np.random.choice(normal_processes, size=len(log_events))
log_events["geo_location"] = np.random.choice(normal_geos, size=len(log_events))

# Overwrite for compromised states
comp_rows = log_events["hidden_compromise_state"] == 1
comp_indices = log_events.index[comp_rows] # Get the exact indices of compromised rows

# 30% chance of malicious process, 15% chance of suspicious, 55% normal
proc_roll = np.random.rand(len(comp_indices))
log_events.loc[comp_indices[proc_roll < 0.30], "process_name"] = np.random.choice(malicious_processes, size=(proc_roll < 0.30).sum())
log_events.loc[comp_indices[(proc_roll >= 0.30) & (proc_roll < 0.45)], "process_name"] = np.random.choice(suspicious_processes, size=((proc_roll >= 0.30) & (proc_roll < 0.45)).sum())

# 40% chance of anomalous geo during compromise
geo_roll = np.random.rand(len(comp_indices))
log_events.loc[comp_indices[geo_roll < 0.40], "geo_location"] = np.random.choice(anomalous_geos, size=(geo_roll < 0.40).sum())

# 4.8 Inject Realism: Missing values in source_ip (3.5%)
source_ips = [f"10.0.{np.random.randint(0,10)}.{np.random.randint(1,255)}" for _ in range(len(log_events))]
log_events["source_ip"] = source_ips
# Set 3.5% to None
missing_mask = np.random.rand(len(log_events)) < 0.035
log_events.loc[missing_mask, "source_ip"] = np.nan

# 4.9 Cleanup and Formatting
log_events["event_id"] = [f"EVT-{str(i).zfill(7)}" for i in range(1, len(log_events) + 1)]
log_events["user_id"] = devices_df.set_index("device_id").loc[log_events["device_id"].values, "user_id"].values

# Assign random event types
log_events["event_type"] = np.random.choice(
    ["Login_Success", "Login_Fail", "File_Download", "File_Upload", "Outbound_Connection", "Process_Start", "Firewall_Block"],
    size=len(log_events)
)

# Select & reorder columns to match schema
log_events = log_events[[
    "event_id", "timestamp", "device_id", "user_id", "event_type", 
    "source_ip", "bytes_transferred", "geo_location", "process_name",
    "hidden_compromise_state", "hidden_episode_type" # KEEP hidden for now to generate alerts later
]]

print(f"Generated {len(log_events):,} log events.")
display(log_events.head())

Initializing hidden compromise state timeline...
✅ Seeded 15 compromise episodes.
Generating log events (this may take a few seconds)...
Generating 7,424,440 individual log rows...
Generated 7,424,440 log events.


,event_id,timestamp,device_id,user_id,event_type,source_ip,bytes_transferred,geo_location,process_name,hidden_compromise_state,hidden_episode_type
0,EVT-0000001,2026-06-01 00:34:08.476726049,DEV-0001,USR-0241,File_Download,10.0.5.27,4.294897e+04,DE,excel.exe,0,0
1,EVT-0000002,2026-06-01 00:28:48.300638502,DEV-0001,USR-0241,Outbound_Connection,10.0.5.166,2.620215e+05,SG,svchost.exe,0,0
2,EVT-0000003,2026-06-01 00:13:22.573525695,DEV-0001,USR-0241,Process_Start,10.0.5.147,1.204164e+08,DE,outlook.exe,0,0
3,EVT-0000004,2026-06-01 00:09:11.549203588,DEV-0001,USR-0241,Outbound_Connection,10.0.6.220,9.713125e+04,US,teams.exe,0,0
4,EVT-0000005,2026-06-01 00:33:46.101856061,DEV-0001,USR-0241,Login_Success,10.0.4.74,8.868615e+04,PH,chrome.exe,0,0


In [8]:
print("Generating alerts table...")

lambda_noise = 0.05
lambda_signal = 0.40


alert_lambda_matrix = lambda_noise + (lambda_signal * C_matrix)

alert_count_matrix = np.random.poisson(alert_lambda_matrix)
total_alerts = alert_count_matrix.sum()
print(f"Generating {total_alerts:,} alerts...")


alert_device_indices = np.repeat(np.arange(n_devices), alert_count_matrix.sum(axis=1))
alert_hour_indeces = np.concatenate([np.repeat(np.arange(total_hours), alert_count_matrix[i]) for i in range(n_devices)])

alerts= pd.DataFrame({
    "device_idx": alert_device_indices,
    "hour_offset": alert_hour_indeces  
})


alerts["device_id"] = device_ids[alerts["device_idx"].values]
alerts["hidden_compromise_state"] = C_matrix[alerts["device_idx"].values, alerts["hour_offset"].values]
alerts["hidden_episode_type"] = episode_matrix[alerts["device_idx"].values, alerts["hour_offset"].values]

alerts["hidden_true_verdict"] = np.where(
    alerts["hidden_compromise_state"] == 0,
    "False_Positive",
    np.where(
        alerts["hidden_episode_type"] == 1,
        "True_Positive",
        "Benign True_Positive"
    )
)


base_seconds = alerts["hour_offset"] * 3600
jitter = np.random.uniform(0, 3600, size=len(alerts))

alerts["timestamp"] = start_date + pd.to_timedelta(base_seconds + jitter, unit="s")


is_noise = alerts["hidden_true_verdict"] == "False_Positive"
is_signal = ~is_noise


alerts["alert_source"] = np.random.choice(["Firewall", "EDR", "IDS", "Email_Gateway"], size=len(alerts))

alerts["confidence_score"] = 0.0
alerts.loc[is_noise, "confidence_score"] = np.random.beta(2, 6, size=is_noise.sum())
alerts.loc[is_signal, "confidence_score"] = np.random.beta(6, 2, size=is_signal.sum())

# Severity (Conditional Probabilities)
# Noise: Low(0.55), Med(0.30), High(0.13), Crit(0.02)
# Signal: Low(0.05), Med(0.20), High(0.45), Crit(0.30)
alerts["severity"] = "Low"
alerts.loc[is_noise, "severity"] = np.random.choice(["Low", "Medium", "High", "Critical"], size=is_noise.sum(), p=[0.55, 0.30, 0.13, 0.02])
alerts.loc[is_signal, "severity"] = np.random.choice(["Low", "Medium", "High", "Critical"], size=is_signal.sum(), p=[0.05, 0.20, 0.45, 0.30])


# Alert Rule & MITRE Tactic
noise_rules = ["Port_Scan_Detected", "Unusual_Process", "Impossible_Travel"]
signal_rules = ["Malware_Signature", "Data_Exfil_Suspected", "Brute_Force"]

noise_tactics = ["Reconnaissance", "Initial_Access", "Execution"]
signal_tactics = ["Lateral_Movement", "Exfiltration", "Command_and_Control"]

alerts["alert_rule_triggered"] = np.random.choice(noise_rules, size=len(alerts))
alerts.loc[is_signal, "alert_rule_triggered"] = np.random.choice(signal_rules, size=is_signal.sum())

alerts["mitre_tactic"] = np.random.choice(noise_tactics, size=len(alerts))
alerts.loc[is_signal, "mitre_tactic"] = np.random.choice(signal_tactics, size=is_signal.sum())

# 5.7 Formatting
alerts["alert_id"] = [f"ALR-{str(i).zfill(5)}" for i in range(1, len(alerts) + 1)]
alerts = alerts[[
    "alert_id", "timestamp", "device_id", "alert_rule_triggered", 
    "severity", "mitre_tactic", "confidence_score", "alert_source",
    "hidden_true_verdict" # Keep hidden for now to generate outcomes later
]]

print(f"Generated {len(alerts):,} alerts.")
display(alerts.head())



Generating alerts table...
Generating 37,080 alerts...
Generated 37,080 alerts.


,alert_id,timestamp,device_id,alert_rule_triggered,severity,mitre_tactic,confidence_score,alert_source,hidden_true_verdict
0,ALR-00001,2026-06-01 10:19:31.168977884,DEV-0001,Unusual_Process,Low,Execution,0.090040,Firewall,False_Positive
1,ALR-00002,2026-06-01 22:13:13.462197773,DEV-0001,Port_Scan_Detected,Low,Execution,0.261239,EDR,False_Positive
2,ALR-00003,2026-06-01 23:40:56.834766516,DEV-0001,Port_Scan_Detected,Medium,Reconnaissance,0.090266,Email_Gateway,False_Positive
3,ALR-00004,2026-06-02 21:33:17.320920415,DEV-0001,Port_Scan_Detected,Low,Initial_Access,0.222366,Email_Gateway,False_Positive
4,ALR-00005,2026-06-03 14:21:59.211391760,DEV-0001,Impossible_Travel,Medium,Initial_Access,0.245826,Firewall,False_Positive


In [ ]:

print("Generating response actions and alert outcomes...")

# 6.1 Setup Analysts & Shifts
analyst_pool = [f"SOC_T1_{name}" for name in ["Alice", "Bob", "Charlie", "Diana"]] + \
               [f"SOC_T2_{name}" for name in ["Eve", "Frank"]]

# Use .copy() to avoid SettingWithCopyWarning, and explicitly create the dataframe
response_actions = pd.DataFrame()
response_actions["alert_id"] = alerts["alert_id"]
response_actions["timestamp"] = alerts["timestamp"]
response_actions["device_id"] = alerts["device_id"]
response_actions["hidden_true_verdict"] = alerts["hidden_true_verdict"]

# Assign Analysts
response_actions["analyst_id"] = np.random.choice(analyst_pool, size=len(response_actions))

# Determine Shift (Day: 06:00-18:00, Night: 18:00-06:00)
alert_hour = response_actions["timestamp"].dt.hour
response_actions["shift_time"] = np.where((alert_hour >= 6) & (alert_hour < 18), "Day_Shift", "Night_Shift")

# 6.2 Calculate Hidden Analyst Fatigue (F_i)
# Sort by analyst and time so we can do a running count of alerts handled
response_actions = response_actions.sort_values(by=["analyst_id", "timestamp"]).reset_index(drop=True)

# Count alerts handled today by this analyst (H_i)
response_actions["alert_date"] = response_actions["timestamp"].dt.date
response_actions["alerts_handled_today"] = response_actions.groupby(["analyst_id", "alert_date"]).cumcount() + 1

# Calculate Fatigue Score: F_i = min(1.0, H_i / 50)
response_actions["hidden_analyst_fatigue"] = (response_actions["alerts_handled_today"] / 50).clip(upper=1.0)

# 6.3 Generate Response Time (Log-Normal mean shifts with fatigue)
# mu_new = 2.0 + 1.5 * F_i
mu_resp = 2.0 + (1.5 * response_actions["hidden_analyst_fatigue"])
response_actions["time_to_respond_mins"] = np.random.lognormal(mean=mu_resp, sigma=0.8)

# 6.4 Generate Action Taken
# Tired analysts are more likely to just close things to clear the queue
actions_fp = ["Close_False_Positive", "Block_IP", "Escalate_to_Tier2"]
actions_tp = ["Isolate_Host", "Reset_Credentials", "Block_IP", "Escalate_to_Tier2"]

is_true_verdict = response_actions["hidden_true_verdict"] != "False_Positive"
response_actions["action_taken"] = "Close_False_Positive" # Default
response_actions.loc[~is_true_verdict, "action_taken"] = np.random.choice(actions_fp, size=(~is_true_verdict).sum(), p=[0.85, 0.10, 0.05])
response_actions.loc[is_true_verdict, "action_taken"] = np.random.choice(actions_tp, size=is_true_verdict.sum(), p=[0.40, 0.30, 0.20, 0.10])


alert_outcomes = response_actions[["alert_id", "device_id", "hidden_true_verdict", "hidden_analyst_fatigue"]].copy()

# 7.1 The Label Corruption Math
# P(Mislabel TP -> FP) = 0.05 + 0.75 * F_i
# P(Mislabel FP -> TP) = 0.01 + 0.05 * F_i
misslabel_roll = np.random.rand(len(alert_outcomes))

# Calculate probabilities
p_tp_to_fp = 0.05 + (0.75 * alert_outcomes["hidden_analyst_fatigue"])
p_fp_to_tp = 0.01 + (0.05 * alert_outcomes["hidden_analyst_fatigue"])

# Apply corruption
alert_outcomes["final_verdict"] = alert_outcomes["hidden_true_verdict"]

# TP -> FP
tp_to_fp_mask = (alert_outcomes["hidden_true_verdict"].isin(["True_Positive", "Benign_True_Positive"])) & (misslabel_roll < p_tp_to_fp)
alert_outcomes.loc[tp_to_fp_mask, "final_verdict"] = "False_Positive"

# FP -> TP
fp_to_tp_mask = (alert_outcomes["hidden_true_verdict"] == "False_Positive") & (misslabel_roll < p_fp_to_tp)
alert_outcomes.loc[fp_to_tp_mask, "final_verdict"] = "True_Positive"

# 7.2 Business Impact (LogNormal * Criticality for True Positives)
# Merge criticality from devices table safely
alert_outcomes = alert_outcomes.merge(devices_df[["device_id", "criticality_score"]], on="device_id", how="left")

# Base financial impact LogNormal(mu=14, sigma=1.5) -> mean ~$3.7M
base_impact = np.random.lognormal(mean=14, sigma=1.5, size=len(alert_outcomes))

# Scale by criticality, only for True Positives that weren't mislabeled
is_labeled_tp = alert_outcomes["final_verdict"] == "True_Positive"
alert_outcomes["business_impact_usd"] = 0.0
alert_outcomes.loc[is_labeled_tp, "business_impact_usd"] = base_impact[is_labeled_tp] * (alert_outcomes.loc[is_labeled_tp, "criticality_score"] / 5.0)

# 7.3 Cleanup Response Actions Table
response_actions_clean = response_actions[[
    "alert_id", "analyst_id", "shift_time", "alerts_handled_today", 
    "action_taken", "time_to_respond_mins"
]].copy()

# Add action_id
response_actions_clean.insert(0, "action_id", [f"ACT-{str(i).zfill(5)}" for i in range(1, len(response_actions_clean) + 1)])

# Final Alert Outcomes Table
alert_outcomes_clean = alert_outcomes[["alert_id", "final_verdict", "business_impact_usd"]].copy()

print(" Generated Response Actions and Alert Outcomes!")
display(response_actions_clean.head())
display(alert_outcomes_clean.head())

# Print the final class imbalance to prove the math worked!
print("\n--- Final Verdict Distribution (Proof of 95/3/2 Imbalance) ---")
print(alert_outcomes_clean["final_verdict"].value_counts(normalize=True) * 100)

Generating response actions and alert outcomes...
✅ Generated Response Actions and Alert Outcomes!


,action_id,alert_id,analyst_id,shift_time,alerts_handled_today,action_taken,time_to_respond_mins
0,ACT-00001,ALR-06875,SOC_T1_Alice,Night_Shift,1,Escalate_to_Tier2,8.023855
1,ACT-00002,ALR-34601,SOC_T1_Alice,Night_Shift,2,Close_False_Positive,8.025607
2,ACT-00003,ALR-33006,SOC_T1_Alice,Night_Shift,3,Close_False_Positive,20.346324
3,ACT-00004,ALR-20475,SOC_T1_Alice,Night_Shift,4,Close_False_Positive,32.455064
4,ACT-00005,ALR-13233,SOC_T1_Alice,Night_Shift,5,Close_False_Positive,5.267937


,alert_id,final_verdict,business_impact_usd
0,ALR-06875,False_Positive,0.0
1,ALR-34601,False_Positive,0.0
2,ALR-33006,False_Positive,0.0
3,ALR-20475,False_Positive,0.0
4,ALR-13233,False_Positive,0.0



--- Final Verdict Distribution (Proof of 95/3/2 Imbalance) ---
final_verdict
False_Positive          93.403452
True_Positive            5.889968
Benign True_Positive     0.706580
Name: proportion, dtype: float64


In [ ]:
print("Cleaning hidden variables and exporting to CSV...")

# 8.1 Drop hidden columns from log_events
# We use .drop() with errors='ignore' so it doesn't crash if a column is already missing
log_events_clean = log_events.drop(columns=["hidden_compromise_state", "hidden_episode_type"], errors='ignore')

# Drop hidden columns from alerts
alerts_clean = alerts.drop(columns=["hidden_true_verdict"], errors='ignore')

# 8.2 Ensure output directory exists
output_dir = CONFIG["output_dir"]
os.makedirs(output_dir, exist_ok=True)

# 8.3 Export all 6 tables to CSV
tables_to_export = {
    "users.csv": users_df,
    "devices.csv": devices_df,
    "log_events.csv": log_events_clean,
    "alerts.csv": alerts_clean,
    "response_actions.csv": response_actions_clean,
    "alert_outcomes.csv": alert_outcomes_clean
}

for filename, df in tables_to_export.items():
    filepath = os.path.join(output_dir, filename)
    df.to_csv(filepath, index=False)
    print(f" Exported {filename} ({len(df):,} rows)")

print("\n DATASET GENERATION COMPLETE!")